# Stage 1 — TF-IDF референс (быстро, только для сравнения)

**Цель:** нижний bound для таблицы метрик. Не идёт в дальнейший pipeline.

**Время:** ~20 минут

**Артефакт:** `reports/stage1_baseline/metrics.json`

**Ожидаемый результат:** accuracy ~0.63–0.65, macro-F1 ~0.62–0.64

## 1.1 Загрузка и обучение

In [17]:
import sys
from pathlib import Path

import pandas as pd
import json
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.config import (
    PROCESSED_DATA_DIR, STAGE1_REPORTS_DIR,
    COL_COMBINED_TEXT, TARGET, TARGET_NAMES_REPORT, RANDOM_STATE,
)
from utils.data_loader import build_combined_text

train_df = pd.read_parquet(Path(PROCESSED_DATA_DIR) / 'train_baseline.parquet')
val_df   = pd.read_parquet(Path(PROCESSED_DATA_DIR) / 'val_baseline.parquet')

# COL_COMBINED_TEXT не хранится в parquet — пересобираем на лету
train_df[COL_COMBINED_TEXT] = train_df.apply(build_combined_text, axis=1)
val_df[COL_COMBINED_TEXT]   = val_df.apply(build_combined_text, axis=1)

assert set(train_df[TARGET].unique()) == {0, 1}

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1, 2),
                              sublinear_tf=True, min_df=3)),
    ('clf',   LogisticRegression(max_iter=1000, C=1.0,
                                 class_weight='balanced',
                                 random_state=RANDOM_STATE, solver='lbfgs')),
])
pipe.fit(train_df[COL_COMBINED_TEXT].values, train_df[TARGET].values)

y_pred = pipe.predict(val_df[COL_COMBINED_TEXT].values)
acc    = accuracy_score(val_df[TARGET], y_pred)
f1     = f1_score(val_df[TARGET], y_pred, average='macro')

print(f'TF-IDF  Val Accuracy: {acc:.4f}  Macro-F1: {f1:.4f}')
print(classification_report(val_df[TARGET], y_pred, target_names=TARGET_NAMES_REPORT))

TF-IDF  Val Accuracy: 0.6367  Macro-F1: 0.6366
              precision    recall  f1-score   support

  Irrelevant       0.61      0.65      0.63      2176
    Relevant       0.66      0.62      0.64      2382

    accuracy                           0.64      4558
   macro avg       0.64      0.64      0.64      4558
weighted avg       0.64      0.64      0.64      4558



## 1.2 Сохранение отчёта

Модель не сохраняется на диск — не нужна в Stage 3/4/5.

In [18]:
report_dir = Path(STAGE1_REPORTS_DIR)
report_dir.mkdir(parents=True, exist_ok=True)

with open(report_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model':        'TF-IDF + LogisticRegression (reference only)',
        'val_accuracy': round(float(acc), 4),
        'val_macro_f1': round(float(f1), 4),
        'note':         'Lower bound. Not used in downstream pipeline.',
    }, f, indent=2, ensure_ascii=False)

print(f'Saved → {report_dir / "metrics.json"}')

Saved → D:\My_courses\NLP_ODS_2026\yandex_relevance\reports\stage1_baseline\metrics.json


## Альтернатива: запуск через скрипт

```bash
python scripts/run_stage1.py
```

In [ ]:
# from utils.stage1_baseline import run_stage1
# run_stage1()